# latex-task2 · De novo drug-design tables → LaTeX

Vina, PoseCheck / PoseBusters, strain, interaction, and reference-ligand similarity from `results/reports/results_drug_design.html`.

## Setup

In [ ]:
import sys
from pathlib import Path
def _find_repo_root():
    for c in [Path.cwd(), *Path.cwd().parents]:
        if (c / "voxbind" / "dataset").is_dir():
            return c
    fb = Path("/home/shpark/prj-denovo/VoxBind")
    if (fb / "voxbind" / "dataset").is_dir():
        return fb
    raise FileNotFoundError("repo root (voxbind/dataset) not found from cwd")
_REPO = _find_repo_root()
sys.path.insert(0, str(_REPO / "notebook" / "results"))
from bs4 import BeautifulSoup, Tag
from latex_common import *   # shared HTML->LaTeX helpers + generic table_to_latex

# ── task2 config (de novo drug design report) ──
DRUG_DESIGN_HTML_PATH = Path("results/reports/results_drug_design.html")
REFERENCE_SIMILARITY_HTML_PATH = Path("notebook/html/reference_similarity_table.html")
REFERENCE_SIMILARITY_EXCLUDE_TAGS = True
REFERENCE_SIMILARITY_TWO_DECIMALS = False
KEEP_BADGES = True
RESIZE_WIDE = True


## Converters (drug design)

In [ ]:
def de_novo_to_latex(table: Tag) -> str:
    """Structure-based drug design (VoxBind) table. Row order, method naming, commented
    rows and placeholders follow a fixed spec; all VALUES are pulled live from the de novo
    CrossDocked table in results.html (Score/Min/Dock/High aff. = Avg / Med; QED/SA/Div = Avg)."""
    rows = [r for r in table.select("tbody > tr")
            if len(r.find_all(["th", "td"], recursive=False)) == 19]

    def find(pred):
        for r in rows:
            cells = r.find_all(["th", "td"], recursive=False)
            if pred(cells[0].get_text(" ", strip=True)):
                return cells
        raise ValueError("de novo row not found for spec predicate")

    def clean(cell):
        t = cell.get_text(strip=True).replace("−", "-").replace("—", "---")
        return t if t else "---"

    def vals(cells):  # 7 template columns: Score/Min/Dock/High (Avg / Med) + QED/SA/Div (Avg)
        pair = lambda a, b: f"{clean(cells[a])} / {clean(cells[b])}"
        return [pair(1, 2), pair(3, 4), pair(5, 6), pair(7, 8),
                clean(cells[9]), clean(cells[11]), clean(cells[13])]

    def row(display, pred, comment=False):
        line = " & ".join([display, *vals(find(pred))]) + r" \\"
        return ("% " + line) if comment else line

    body_rows = [
        row("Reference", lambda t: t.startswith("Reference")),
        r"\midrule",
        row("AR", lambda t: t.startswith("AR ")),
        row("Pocket2Mol", lambda t: t.startswith("Pocket2Mol")),
        row("DiffSBDD", lambda t: t.startswith("DiffSBDD")),
        row("TargetDiff", lambda t: t.startswith("TargetDiff")),
        row(r"DecompDiff\textsuperscript{$\dagger$}",
            lambda t: t.startswith("DecompDiff†"), comment=True),
        row(r"VoxBind\textsubscript{\scriptsize $\sigma$=0.9}",
            lambda t: t.startswith("VoxBind σ=0.9"), comment=True),
        row(r"VoxBind\textsubscript{\scriptsize $\sigma$=1.0}",
            lambda t: t.startswith("VoxBind σ=1.0"), comment=True),
        r"% \midrule",
        row(r"DecompDiff\textsubscript{\scriptsize ref-informed}",
            lambda t: t.startswith("DecompDiff reproduced ref-informed")),
        row(r"DecompDiff\textsubscript{\scriptsize ref-free}",
            lambda t: t.startswith("DecompDiff reproduced ref-free")),
        row(r"VoxBind\textsubscript{\scriptsize $\sigma$=0.9}",
            lambda t: t.startswith("VoxBind reproduced") and "σ = 0.9" in t),
        row(r"VoxBind\textsubscript{\scriptsize $\sigma$=1.0}",
            lambda t: t.startswith("VoxBind reproduced") and "σ = 1.0" in t, comment=True),
        r"Funcbind \\",
        r"\midrule",
        r"\textbf{VoxBind + C} & \multicolumn{7}{c}{?} \\",
        row(r"\textbf{VoxBind + CDG}", lambda t: t.startswith("Ours")),
    ]

    caption = (
        r"\textbf{Structure-based drug design results} on 92 test pockets with experimental density data available of CrossDocked2020 benchmark. Vina and "
        r"high-affinity cells report mean / median; arrows indicate the preferred direction."
    )
    body = [
        r"\begin{table}[!t]",
        r"    \centering",
        r"    \caption{",
        f"        {caption}",
        r"    }",
        r"    \label{tab:result-drug-design}",
        r"    \resizebox{.98\textwidth}{!}{%",
        r"        \begin{tabular}{@{}lcccccccc@{}}",
        r"            \toprule",
        r"            \multirow{2}{*}{\textbf{Method}} & \multicolumn{4}{c}{\textbf{Vina evaluation}} & \multicolumn{4}{c}{\textbf{Sample quality}} \\",
        r"            \cmidrule(lr){2-5}\cmidrule(lr){6-9}",
        r"            & \textbf{Score $\downarrow$} & \textbf{Min $\downarrow$} & \textbf{Dock $\downarrow$} & \textbf{High aff. $\uparrow$} & \textbf{QED $\uparrow$} & \textbf{SA $\uparrow$} & \textbf{Diversity $\uparrow$} & \textbf{\# Atoms / Mol} \\",
        r"            \midrule",
    ]
    body.extend("            " + r for r in body_rows)
    body.extend([
        r"            \bottomrule",
        r"        \end{tabular}",
        r"    }",
        r"\end{table}",
    ])
    return "\n".join(body)


REFERENCE_SIMILARITY_CAPTION = (
    r"\textbf{Reference-ligand similarity} on the CrossDocked benchmark. "
    r"Metrics are averaged over pockets."
)
REFERENCE_SIMILARITY_LABEL = "tab:result-drug-reference-similarity"


def reference_similarity_to_latex(table: Tag, wrap: bool = False) -> str:
    """Transpose reference-ligand similarity: metrics as rows, methods as columns.

    The source is build_reference_similarity.py's fragment, not results.html. That
    script cut the metric set from the seven fingerprints of the 260820 table down to
    the two this literature actually reports — ECFP4/Morgan (radius 2, 2048 bit)
    Tanimoto, and the Bemis-Murcko scaffold match rate. MACCS / AtomPair / RDKit /
    Dice come back under --full and 3D shape under --with-3d, so the groups are read
    off the fragment's two-row header rather than hardcoded: changing the fingerprint
    set again does not touch this function.

    Layout: two stub columns, a metric group name and its statistic. A group with
    several statistics (ECFP4 -> mean / median) gets a \multirow name; a
    single-statistic group (Scaffold match) spans both stub columns with its name
    broken over two lines. \shortstack is plain LaTeX, so this needs only booktabs
    and multirow — no makecell.

    wrap=True emits the 260820 layout instead: a \resizebox'd wraptable for sitting
    beside the body text (needs wrapfig), same rows either way.

    Both outputs are byte-identical to the .tex files the builder writes
    (reference_similarity.tex and _wrap.tex); the two entry points exist so a
    regression in either is a one-line diff to catch.
    """
    def plain(cell) -> str:
        text = cell_to_latex(cell, keep_badges=not REFERENCE_SIMILARITY_EXCLUDE_TAGS)
        bold_match = re.fullmatch(r"\\textbf\{([^{}]+)\}", text)
        return bold_match.group(1) if bold_match else text

    header_rows = table.select("thead > tr")
    if not header_rows:
        raise ValueError("reference-similarity 표에 <thead>가 없습니다.")
    top = header_rows[0].find_all("th", recursive=False)[1:]   # 첫 칸은 Method stub
    subs = header_rows[1].find_all("th", recursive=False) if len(header_rows) > 1 else []

    groups, sub_index = [], 0
    for cell in top:
        span = int(cell.get("colspan", 1))
        if span > 1:
            groups.append((plain(cell), [plain(s) for s in subs[sub_index:sub_index + span]]))
            sub_index += span
        else:
            groups.append((plain(cell), [""]))   # rowspan=2, 통계 이름이 따로 없는 지표
    if not groups:
        raise ValueError("reference-similarity 표의 헤더에서 지표 열을 찾지 못했습니다.")
    n_values = sum(len(stats) for _, stats in groups)

    methods, values_by_method = [], []
    for row in table.select("tbody > tr"):
        cells = row.find_all(["th", "td"], recursive=False)
        if len(cells) != n_values + 1:
            continue
        methods.append(plain(cells[0]))
        values_by_method.append([
            format_similarity_two_decimal_places(cell_to_latex(cell, keep_badges=False))
            if REFERENCE_SIMILARITY_TWO_DECIMALS
            else cell_to_latex(cell, keep_badges=False)
            for cell in cells[1:]
        ])
    if not methods:
        raise ValueError(
            f"값 {n_values}개짜리 method 행이 없습니다. "
            "fragment를 build_reference_similarity.py로 다시 만들어 주세요."
        )

    pad = "        " if wrap else "    "        # \begin{tabular} 들여쓰기
    line = pad + "    "                        # 그 안의 행
    body = [
        r"\begin{wraptable}{r}{0.55\textwidth}" if wrap else r"\begin{table}[t]",
        r"    \centering",
        r"    \caption{",
        "        " + REFERENCE_SIMILARITY_CAPTION,
        r"    }",
        rf"    \label{{{REFERENCE_SIMILARITY_LABEL}}}",
    ]
    if wrap:
        body.append(r"    \resizebox{.98\linewidth}{!}{%")
    body += [
        pad + rf"\begin{{tabular}}{{@{{}}ll{'c' * len(methods)}@{{}}}}",
        line + r"\toprule",
        line + " & ".join([r"\multicolumn{2}{@{}l}{\textbf{Method}}"]
                          + [rf"\textbf{{{method}}}" for method in methods]) + r" \\",
        line + r"\midrule",
    ]
    value_index = 0
    for group_index, (name, stats) in enumerate(groups):
        if group_index:
            body.append(line + r"\addlinespace")
        single = len(stats) == 1 and not stats[0]
        for stat_index, stat in enumerate(stats):
            values = [row[value_index] for row in values_by_method]
            value_index += 1
            if single:
                head = (r"\multicolumn{2}{@{}l}{\shortstack[l]{"
                        + r"\\ ".join(name.split(" ", 1)) + "}}")
            elif stat_index == 0:
                head = rf"\multirow{{{len(stats)}}}{{*}}{{{name}}} & {stat}"
            else:
                head = f" & {stat}"
            body.append(line + " & ".join([head, *values]) + r" \\")
    body += [line + r"\bottomrule", pad + r"\end{tabular}"]
    if wrap:
        body.append(r"    }")
    body.append(r"\end{wraptable}" if wrap else r"\end{table}")
    return "\n".join(body)


def drug_design_vina_to_latex(table: Tag) -> str:
    """Table 1 of results_drug_design.html -> LaTeX.

    That table is the live one for de novo generation, so the Vina numbers are taken
    from it rather than from results.html (whose de novo block is a different, older
    pocket set). Layout: Method + Score/Min/Dock as Avg|Med pairs + High aff. + QED +
    SA + Div + heavy atoms + n, i.e. 13 cells per data row. Section rows carry a single
    cell and are turned into \midrule; rows still reading TBA are emitted commented out
    so the template keeps its shape while a baseline is still sampling.
    """
    body = table.select("tbody > tr")

    def clean(cell: Tag) -> str:
        t = cell.get_text(" ", strip=True)
        t = t.replace("−", "-").replace("—", "---").replace("–", "--")
        return t if t else "---"

    def display_name(cell: Tag) -> str:
        """Method label without the tag chips and sub-line the HTML carries."""
        c = cell.__copy__()
        for junk in c.select(".tag, .sub, .nsub, span[style]"):
            junk.decompose()
        name = c.get_text(" ", strip=True)
        return {
            "Reference ligand": "Reference",
            "VoxBind σ=0.9": r"VoxBind\textsubscript{\scriptsize $\sigma$=0.9}",
            "Ours · v1": r"Ours\textsubscript{\scriptsize v1}",
            "Ours · v2": r"Ours\textsubscript{\scriptsize v2}",
            "Ours · v3": r"Ours\textsubscript{\scriptsize v3}",
        }.get(name, escape_latex_text(name))

    lines, n_tba = [], 0
    for r in body:
        cells = r.find_all(["th", "td"], recursive=False)
        if len(cells) == 1:                       # section divider
            if lines and lines[-1] != r"\midrule":
                lines.append(r"\midrule")
            continue
        if len(cells) != 16:
            continue
        vals = [clean(c) for c in cells[1:]]
        line = " & ".join([display_name(cells[0]), *vals]) + r" \\"
        if any(v == "TBA" for v in vals):         # still sampling
            n_tba += 1
            line = "% " + line
        lines.append(line)

    title = table.find_previous("p", class_="table-title")
    title = title.get_text(" ", strip=True) if title else "Vina affinity"
    caption = (r"\textbf{De novo drug design on CrossDocked.} "
               + escape_latex_text(title.split("·", 1)[-1].strip())
               + r". Vina Score / Min / Dock, QED and SA are reported as mean\,/\,med over all "
                 r"generated molecules pooled across pockets -- the convention DecompDiff, "
                 r"VoxBind and TargetDiff state, and the one every row here follows. Diversity "
                 r"is the mean pairwise Tanimoto within a pocket, so its pair is over the 79 "
                 r"pocket values, and High aff.\ is the per-pocket share of molecules "
                 r"out-docking that pocket's reference ligand, averaged over pockets.")

    # Three header rows, laid out like baseline.html: the two evaluation blocks on top,
    # the metric names under them, mean/med under each metric that has both. Columns are
    # 1 Method | 2-8 Vina (Score/Min/Dock + High aff.) | 9-14 QED/SA/Div | 15 heavy atoms | 16 n.
    header = [
        r"\begin{tabular}{l" + "rr" * 3 + "r" + "rr" * 3 + "rr}",
        r"\toprule",
        r"\multirow{3}{*}{Method} & \multicolumn{7}{c}{Vina evaluation}"
        r" & \multicolumn{8}{c}{Sample quality} \\",
        r"\cmidrule(lr){2-8}\cmidrule(lr){9-16}",
        r" & \multicolumn{2}{c}{Score $\downarrow$}"
        r" & \multicolumn{2}{c}{Min $\downarrow$}"
        r" & \multicolumn{2}{c}{Dock $\downarrow$}"
        r" & \multirow{2}{*}{\shortstack{High aff.\\ \%\,$\uparrow$}}"
        r" & \multicolumn{2}{c}{QED $\uparrow$}"
        r" & \multicolumn{2}{c}{SA $\uparrow$}"
        r" & \multicolumn{2}{c}{Div. $\uparrow$}"
        r" & \multirow{2}{*}{\shortstack{Heavy\\ atoms}}"
        r" & \multirow{2}{*}{$n$} \\",
        r"\cmidrule(lr){2-3}\cmidrule(lr){4-5}\cmidrule(lr){6-7}"
        r"\cmidrule(lr){9-10}\cmidrule(lr){11-12}\cmidrule(lr){13-14}",
        r" & mean & med & mean & med & mean & med & & mean & med & mean & med"
        r" & mean & med & & \\",
        r"\midrule",
    ]

    # Indented the way the manuscript template is written -- table body one level in,
    # tabular contents three -- so the block pastes straight into the paper source.
    def ind(rows, level):
        return [INDENT * level + r for r in rows]

    out = ([r"\begin{table}[t]"]
           + ind([r"\centering",
                  r"\caption{" + caption + "}",
                  r"\label{tab:denovo-vina}",
                  r"\resizebox{.98\textwidth}{!}{%"], 1)
           + ind(header[:1], 2)
           + ind([*header[1:], *lines, r"\bottomrule"], 3)
           + ind([r"\end{tabular}"], 2)
           + ind([r"}"], 1)
           + (ind([f"% {n_tba} row(s) commented out: still sampling at build time."], 1)
              if n_tba else [])
           + [r"\end{table}"])
    return "\n".join(out)

## Load report + list tables

In [ ]:
dd_path = resolve_html_path(DRUG_DESIGN_HTML_PATH)
dd_soup = BeautifulSoup(dd_path.read_text(encoding="utf-8"), "html.parser")
def _dd_title(tbl):
    t = tbl.find_previous("p", class_="table-title")
    return t.get_text(" ", strip=True) if t else ""
dd_tables = dd_soup.find_all("table")
print(f"Source: {dd_path}  ·  {len(dd_tables)} tables")
for i, t in enumerate(dd_tables):
    print(f"  [{i}] {_dd_title(t)}")


## Vina affinity + sample quality

In [ ]:
# de novo Vina affinity + sample quality
vina = [t for t in dd_tables if "Vina affinity" in _dd_title(t)]
print(drug_design_vina_to_latex(vina[0]))


## Pose quality / PoseBusters / strain / interactions

In [ ]:
# PoseCheck / PoseBusters / strain / interaction tables (generic renderer)
POSE_TITLES = ["Pose quality", "PoseBusters", "Strain energy", "interaction profile"]
for key in POSE_TITLES:
    hit = [t for t in dd_tables if key.lower() in _dd_title(t).lower()]
    if not hit:
        print(f"% (no table titled ~{key!r})\n"); continue
    print(f"% ── {_dd_title(hit[0])} ──")
    print(table_to_latex(hit[0], 99, keep_badges=KEEP_BADGES, resize_wide=RESIZE_WIDE))
    print()


## Reference-ligand similarity

In [ ]:
rs_path = resolve_html_path(REFERENCE_SIMILARITY_HTML_PATH)
rs_table = BeautifulSoup(rs_path.read_text(encoding="utf-8"), "html.parser").find("table")
print(reference_similarity_to_latex(rs_table))
print("\n% ── wrapfig variant ──")
print(reference_similarity_to_latex(rs_table, wrap=True))
